# Monga Qwen3 0.6B — QLoRA SFT smoke run

This notebook trains the first narrow Monga memory-grounding adapter on the synthetic seed dataset.

**Colab:** Runtime → Change runtime type → GPU before running the cells.


In [7]:
!nvidia-smi


Fri Sep 18 04:45:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q -U unsloth trl datasets accelerate


In [9]:
import os, shutil, pathlib
repo_dir = pathlib.Path('/content/Monga')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
!git clone -q -b feat/monga-finetune-sft-seed https://github.com/joker-bot0420/Monga.git /content/Monga
%cd /content/Monga
!git rev-parse --abbrev-ref HEAD
!git rev-parse --short HEAD


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/Monga'
/content/Monga
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [10]:
!python finetune/validate_dataset.py


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'finetune/validate_dataset.py': [Errno 2] No such file or directory


## Train

This is intentionally a tiny smoke run: Qwen3-0.6B, 4-bit QLoRA, 3 epochs, 1024-token context.
The held-out set is not used for gradient updates.


In [11]:
!python finetune/train_sft.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'finetune/train_sft.py': [Errno 2] No such file or directory
[Errno 2] No such file or directory: '/content/Monga'
/content/Monga
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [12]:
from pathlib import Path
adapter = Path('/content/monga-qwen3-0.6b-memory-sft/adapter')
assert adapter.exists(), f'Adapter not found: {adapter}'
print('Adapter files:')
for p in sorted(adapter.iterdir()):
    print(f'  {p.name}: {p.stat().st_size / (1024*1024):.2f} MB')


AssertionError: Adapter not found: /content/monga-qwen3-0.6b-memory-sft/adapter

In [14]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

[Errno 2] No such file or directory: '/content/Monga'
/content/Monga
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [15]:
%cd /content
!rm -rf Monga
!git clone -b feat/monga-finetune-sft-seed https://github.com/joker-bot0420/Monga.git Monga
%cd /content/Monga
!git rev-parse --short HEAD

/content
Cloning into 'Monga'...
remote: Enumerating objects: 2016, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 2016 (delta 168), reused 83 (delta 71), pack-reused 1689 (from 3)
Receiving objects: 100% (2016/2016), 510.76 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (883/883), done.
/content/Monga
9b28314


In [16]:
!rm -rf /content/monga-qwen3-0.6b-memory-sft

In [17]:
!python finetune/train_sft.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Generating train split: 24 examples [00:00, 9965.68 examples/s]
Generating validation split: 6 examples [00:00, 4025.89 examples/s]
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with 

In [18]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 650 bytes | 650.00 KiB/s, done.
From https://github.com/joker-bot0420/Monga
   9b28314..65bdb61  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating 9b28314..65bdb61
Fast-forward
 finetune/train_sft.py | 18 +++++++++++++++++-
 1 file changed, 17 insertions(+), 1 deletion(-)
65bdb61


In [19]:
!rm -rf /content/monga-qwen3-0.6b-memory-sft

In [20]:
!python finetune/train_sft.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Map: 100% 24/24 [00:00<00:00, 3093.52 examples/s]
Map: 100% 6/6 [00:00<00:00, 1263.41 examples/s]
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/

In [21]:
from pathlib import Path

adapter = Path('/content/monga-qwen3-0.6b-memory-sft/adapter')

print("adapter exists:", adapter.exists())

for p in sorted(adapter.iterdir()):
    print(p.name, f"{p.stat().st_size / (1024*1024):.2f} MB")

adapter exists: True
README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 38.55 MB
chat_template.jinja 0.00 MB
tokenizer.json 10.89 MB
tokenizer_config.json 0.00 MB


In [22]:
!cd /content/monga-qwen3-0.6b-memory-sft && zip -qr /content/monga-qwen3-0.6b-memory-sft-adapter.zip adapter

from google.colab import files
files.download('/content/monga-qwen3-0.6b-memory-sft-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.25 KiB | 766.00 KiB/s, done.
From https://github.com/joker-bot0420/Monga
   65bdb61..30ea8e6  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating 65bdb61..30ea8e6
Fast-forward
 finetune/README.md           |   2 +-
 finetune/evaluate_heldout.py | 155 +++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 156 insertions(+), 1 deletion(-)
 create mode 100644 finetune/evaluate_heldout.py
30ea8e6


In [24]:
!python finetune/evaluate_heldout.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== loading: Qwen/Qwen3-0.6B ===
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfx

In [25]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.73 KiB | 1.73 MiB/s, done.
From https://github.com/joker-bot0420/Monga
   30ea8e6..823c16a  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating 30ea8e6..823c16a
Fast-forward
 finetune/evaluate_checkpoints.py | 142 +++++++++++++++++++++++++++++++++++++++
 1 file changed, 142 insertions(+)
 create mode 100644 finetune/evaluate_checkpoints.py
823c16a


## Stop here after the first run

Do not merge/export to GGUF yet. Bring back the training output (especially train/eval loss and any error) so we can verify the smoke run before evaluating held-out behavior.


In [26]:
!python finetune/evaluate_checkpoints.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== loading: /content/monga-qwen3-0.6b-memory-sft/checkpoint-6 ===
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 310/310 [00:00<00:00, 877.64it/s]
Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

In [27]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 48 (delta 1), reused 1 (delta 1), pack-reused 43 (from 1)
Unpacking objects: 100% (48/48), 13.15 KiB | 396.00 KiB/s, done.
From https://github.com/joker-bot0420/Monga
   823c16a..ef9b22a  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating 823c16a..ef9b22a
Fast-forward
 finetune/data/heldout_no_memory_v2.jsonl   |  12 +++
 finetune/data/train.jsonl                  |  36 +++++++
 finetune/data/train_contradiction_v2.jsonl |   8 ++
 finetune/data/train_no_memory_v2.jsonl     |   4 +
 finetune/data/train_no_memory_v2b.jsonl    |   8 ++
 finetune/data/train_partial_v2.jsonl       |   8 ++
 finetune/data/train_unknown_v2.jsonl       |   8 ++
 finetune/data/validation_v2.jsonl          |   6 ++
 finetune/train_sft_v2.py                   | 156 +++++++++++++++++++++++++++++
 finetune/train_sft_v2_final.py      

In [28]:
!rm -rf /content/monga-qwen3-0.6b-memory-sft-v2

!python finetune/train_sft_v2_final.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft-v2 \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Generating train split: 60 examples [00:00, 18128.38 examples/s]
Generating validation split: 12 examples [00:00, 4584.77 examples/s]
v2 train examples: 60
v2 validation examples: 12
Map: 100% 60/60 [00:00<00:00, 4216.65 examples/s]
Map: 100% 12/12 [00:00<00:00, 1516.61 examples/s]
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environm

In [29]:
!cd /content/monga-qwen3-0.6b-memory-sft-v2 && \
zip -qr /content/monga-qwen3-0.6b-memory-sft-v2-adapter.zip adapter

from google.colab import files
files.download('/content/monga-qwen3-0.6b-memory-sft-v2-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.72 KiB | 586.00 KiB/s, done.
From https://github.com/joker-bot0420/Monga
   ef9b22a..3fae221  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating ef9b22a..3fae221
Fast-forward
 finetune/evaluate_v1_v2.py | 125 +++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 125 insertions(+)
 create mode 100644 finetune/evaluate_v1_v2.py
3fae221


In [31]:
!python finetune/evaluate_v1_v2.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== loading: /content/monga-qwen3-0.6b-memory-sft/adapter ===
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 310/310 [00:00<00:00, 721.29it/s]
Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transf

In [32]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 24 (delta 17), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (24/24), 6.72 KiB | 982.00 KiB/s, done.
From https://github.com/joker-bot0420/Monga
   3fae221..b7f5f29  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating 3fae221..b7f5f29
Fast-forward
 finetune/data/heldout_v3.jsonl        |  12 +++
 finetune/data/train_balance_v3a.jsonl |  12 +++
 finetune/data/train_balance_v3b.jsonl |  12 +++
 finetune/data/validation_v3.jsonl     |   6 ++
 finetune/train_sft_v3.py              | 157 ++++++++++++++++++++++++++++++++++
 5 files changed, 199 insertions(+)
 create mode 100644 finetune/data/heldout_v3.jsonl
 create mode 100644 finetune/data/train_balance_v3a.jsonl
 create mode 100644 finetune/data/train_balance_v3b.jsonl
 create mode 100644 finetune/data/validation_v3.jsonl
 create mod

In [33]:
!rm -rf /content/monga-qwen3-0.6b-memory-sft-v3

!python finetune/train_sft_v3.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft-v3 \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Generating train split: 84 examples [00:00, 21342.47 examples/s]
Generating validation split: 18 examples [00:00, 7077.00 examples/s]
v3 train examples: 84
v3 validation examples: 18
Map: 100% 84/84 [00:00<00:00, 6117.86 examples/s]
Map: 100% 18/18 [00:00<00:00, 3187.16 examples/s]
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environm

In [34]:
!cd /content/monga-qwen3-0.6b-memory-sft-v3 && \
zip -qr /content/monga-qwen3-0.6b-memory-sft-v3-adapter.zip adapter

from google.colab import files
files.download('/content/monga-qwen3-0.6b-memory-sft-v3-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.80 KiB | 1.80 MiB/s, done.
From https://github.com/joker-bot0420/Monga
   b7f5f29..c0d228a  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating b7f5f29..c0d228a
Fast-forward
 finetune/evaluate_v1_v2_v3.py | 135 ++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 135 insertions(+)
 create mode 100644 finetune/evaluate_v1_v2_v3.py
c0d228a


In [36]:
!python finetune/evaluate_v1_v2_v3.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== loading: /content/monga-qwen3-0.6b-memory-sft/adapter ===
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 310/310 [00:00<00:00, 960.06it/s]
Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transf

In [37]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 19 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (19/19), 5.70 KiB | 1.14 MiB/s, done.
From https://github.com/joker-bot0420/Monga
   c0d228a..faf45b6  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating c0d228a..faf45b6
Fast-forward
 finetune/data/heldout_v4.jsonl        |  12 +++
 finetune/data/train_polarity_v4.jsonl |  16 ++++
 finetune/data/validation_v4.jsonl     |   8 ++
 finetune/train_sft_v4.py              | 159 ++++++++++++++++++++++++++++++++++
 4 files changed, 195 insertions(+)
 create mode 100644 finetune/data/heldout_v4.jsonl
 create mode 100644 finetune/data/train_polarity_v4.jsonl
 create mode 100644 finetune/data/validation_v4.jsonl
 create mode 100644 finetune/train_sft_v4.py
faf45b6


In [38]:
!rm -rf /content/monga-qwen3-0.6b-memory-sft-v4

!python finetune/train_sft_v4.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft-v4 \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Generating train split: 100 examples [00:00, 10105.05 examples/s]
Generating validation split: 26 examples [00:00, 5268.21 examples/s]
v4 train examples: 100
v4 validation examples: 26
Map: 100% 100/100 [00:00<00:00, 5684.49 examples/s]
Map: 100% 26/26 [00:00<00:00, 3034.45 examples/s]
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` envi

In [39]:
!cd /content/monga-qwen3-0.6b-memory-sft-v4 && \
zip -qr /content/monga-qwen3-0.6b-memory-sft-v4-adapter.zip adapter

from google.colab import files
files.download('/content/monga-qwen3-0.6b-memory-sft-v4-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
%cd /content/Monga
!git pull --ff-only
!git rev-parse --short HEAD

/content/Monga
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.77 KiB | 1.77 MiB/s, done.
From https://github.com/joker-bot0420/Monga
   faf45b6..6607c96  feat/monga-finetune-sft-seed -> origin/feat/monga-finetune-sft-seed
Updating faf45b6..6607c96
Fast-forward
 finetune/evaluate_v3_v4.py | 136 +++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 136 insertions(+)
 create mode 100644 finetune/evaluate_v3_v4.py
6607c96


In [41]:
!python finetune/evaluate_v3_v4.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== loading: /content/monga-qwen3-0.6b-memory-sft-v3/adapter ===
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 310/310 [00:00<00:00, 900.26it/s]
Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/tra

In [42]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/monga-qwen3-0.6b-memory-sft-v4/adapter",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("v4 adapter loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


v4 adapter loaded
